# MLops-Sentinel — Observability Lab

מחברת לימוד פרקטית עבור Phase 4 בפרויקט Sentinel.

המטרה: ללמוד את ארבעת החלקים שנצטרך בפרויקט:
- **Metrics — Prometheus**
- **Traces — OpenTelemetry**
- **Logs — Loki**
- **Dashboards — Grafana**

נעבוד שלב-שלב, וכל פעם נתקדם רק לנושא הבא לאחר שהשלב הקודם ברור ועובד.

## התמונה הגדולה

בפרויקט:
- **Prometheus** אומר לנו *מה קרה* — לדוגמה, ה-latency עלה.
- **OpenTelemetry** עוזר להבין *איפה קרה* — לדוגמה, Redis היה איטי.
- **Loki** עוזר להבין *למה קרה* — לדוגמה, שגיאת connection.
- **Grafana** מרכז את הכל במקום אחד ומציג dashboards וחקירה.

הזרימה שנרצה להגיע אליה:

`FastAPI -> Metrics / Traces / Logs -> Prometheus / Trace backend / Loki -> Grafana`


# שלב 1 — Metrics עם Prometheus

## מה זה Metric?

Metric הוא מספר שנמדד לאורך זמן.

דוגמאות מה-API שלנו:
- כמה בקשות inference קיבלנו.
- כמה בקשות נכשלו.
- כמה זמן inference לקח.
- כמה בקשות בשנייה מגיעות למערכת.

בשלב הראשון עדיין **לא מתקינים Prometheus**.

קודם נלמד לגרום ל-FastAPI שלנו לחשוף endpoint בשם:

`/metrics`

Prometheus יקרא את ה-endpoint הזה בהמשך באמצעות מנגנון שנקרא **scrape**.


## שלושת סוגי ה-Metrics החשובים לנו

### 1. Counter
מספר שרק עולה.

דוגמה:
`inference_requests_total`

מתאים לספירת:
- requests
- errors
- predictions

### 2. Gauge
מספר שיכול לעלות וגם לרדת.

דוגמה:
- מספר requests פעילים כרגע.
- שימוש בזיכרון.

### 3. Histogram
מודד התפלגות של ערכים.

בפרויקט שלנו נשתמש בו למדידת latency:

`inference_latency_seconds`

ה-Histogram חשוב כי בהמשך נוכל לחשב P95 / P99.


In [1]:
from prometheus_client import Counter, Histogram

REQUEST_COUNT = Counter(
    "inference_requests_total",
    "Total inference requests",
    ["status"],
)

LATENCY = Histogram(
    "inference_latency_seconds",
    "Inference request latency in seconds",
)

print("Metrics objects created successfully")


Metrics objects created successfully


## מה עשינו בקוד?

יצרנו שני metrics:

### `REQUEST_COUNT`
Counter שסופר בקשות.

יש לו label בשם `status`, ולכן נוכל לקבל סדרות נפרדות כמו:
- `status="success"`
- `status="error"`

### `LATENCY`
Histogram שמודד כמה זמן בקשה לקחה.

בהמשך FastAPI ימדוד את זמן `/predict` ו-Prometheus ישמור את התוצאה לאורך זמן.


In [2]:
REQUEST_COUNT.labels(status="success").inc()
REQUEST_COUNT.labels(status="success").inc()
REQUEST_COUNT.labels(status="error").inc()

print("Sample requests recorded")


Sample requests recorded


## בדיקה קטנה

אחרי התא הזה:
- success אמור להיות `2`
- error אמור להיות `1`

זה מדגים את עקרון ה-Counter.

בשלב הבא נחבר את אותם metrics ל-FastAPI האמיתי של Sentinel וניצור `/metrics`.


# המשך המחברת

החלקים הבאים יתווספו בהמשך לפי הסדר:

1. **חיבור metrics ל-FastAPI**
2. **הרצת Prometheus ו-scrape של Sentinel API**
3. **PromQL בסיסי**
4. **Grafana + Prometheus Data Source**
5. **Golden Signals Dashboard**
6. **OpenTelemetry Traces**
7. **Loki + Structured JSON Logs**
8. **Correlation: Metric -> Trace -> Log**
